# Estadísticas de las capturas de ríos

Una captura en 4K por río, sacada del video SVS 14990 (SWOT). Elegí en cada río el mes en que el cauce principal se ve más nítido. En casi todos coincidió con caudal bajo, así que el río aparece mayormente en blanco.

| Archivo | Segundo del video | Mes |
|---|---|---|
| mississippi_2023-10.png | 69.93 | octubre 2023 |
| amazonas_2023-11.png | 115.28 | noviembre 2023 |
| congo_2024-09.png | 164.42 | septiembre 2024 |
| ganges_2024-03.png | 186.95 | marzo 2024 |
| yangtse_2023-11.png | 230.45 | noviembre 2023 |

El notebook está en estado de prototipo. Ya lee la escala de la leyenda desde la propia imagen, detecta los píxeles de río, les asigna un valor de volumen entre 0 (lower) y 1 (higher) y calcula estadísticas básicas. Los umbrales todavía hay que ajustarlos.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi

CARPETA_CAPTURAS = Path('capturas')
capturas = sorted(CARPETA_CAPTURAS.glob('*.png'))
[c.name for c in capturas]

## Ver las capturas

In [ ]:
fig, ejes = plt.subplots(2, 3, figsize=(18, 7))
for eje, ruta in zip(ejes.flat, capturas):
    eje.imshow(Image.open(ruta)); eje.set_title(ruta.stem); eje.axis('off')
ejes.flat[-1].axis('off')
plt.tight_layout()

## Zonas fijas del cuadro

Todas las capturas comparten el mismo diseño, con 3840 x 2160 píxeles. La barra de la leyenda va de x 2680 a 3740 y de y 1990 a 2020. La leyenda completa ocupa la esquina inferior derecha y la fecha la esquina inferior izquierda, así que esas dos zonas se excluyen del análisis.

In [ ]:
LEYENDA = dict(x0=2680, x1=3740, y0=1990, y1=2020)
ZONA_LEYENDA = (slice(1900, None), slice(2600, None))
ZONA_FECHA = (slice(1950, None), slice(None, 760))

def leer_paleta(imagen, cantidad=64):
    """Toma la barra de la leyenda y devuelve sus colores de izquierda (lower) a derecha (higher)."""
    franja = imagen[LEYENDA['y0']:LEYENDA['y1'], LEYENDA['x0']:LEYENDA['x1']].astype(float).mean(0)
    nucleo = np.ones(21) / 21  # suaviza las marquitas verticales de la barra
    suave = np.stack([np.convolve(franja[:, i], nucleo, 'same') for i in range(3)], 1)[10:-10]
    indices = np.linspace(0, len(suave) - 1, cantidad).astype(int)
    return suave[indices], np.linspace(0, 1, cantidad)

imagen = np.asarray(Image.open(capturas[0]).convert('RGB'))
paleta, valores = leer_paleta(imagen)
plt.figure(figsize=(10, 1)); plt.imshow(paleta[None].astype(np.uint8), aspect='auto'); plt.yticks([])
plt.xticks([0, 32, 63], ['lower', 'normal', 'higher']); plt.title('Escala leída de la leyenda');

## Detección de píxeles de río

1. El fondo local se estima con una apertura morfológica de 11 x 11. Los ríos son líneas finas y más claras que su entorno, así que la resta entre la luminancia y su apertura (top-hat) los resalta.
2. Se descartan el espacio negro, la nieve y las nubes (zonas claras grandes) y el texto con el nombre del río (manchas claras, neutras y gruesas).
3. Para darle un valor a cada píxel de río se compara la dirección de color *píxel menos fondo* con la de *color de la paleta menos fondo* y se elige el color de la paleta más parecido. Así se tiene en cuenta que las líneas finas se mezclan con el fondo.

In [ ]:
def analizar(ruta, umbral_tophat=22):
    imagen = np.asarray(Image.open(ruta).convert('RGB')).astype(np.float32)
    alto, ancho, _ = imagen.shape
    paleta, valores = leer_paleta(imagen)

    luminancia = imagen @ np.array([0.299, 0.587, 0.114], np.float32)
    fondo = np.stack([ndi.grey_opening(imagen[..., c], size=(11, 11)) for c in range(3)], -1)
    tophat = luminancia - ndi.grey_opening(luminancia, size=(11, 11))

    util = np.ones((alto, ancho), bool)
    util[ZONA_LEYENDA] = False
    util[ZONA_FECHA] = False
    util &= luminancia > 20                                            # espacio
    util &= ndi.grey_opening(luminancia, size=(31, 31)) < 165          # nieve y nubes
    neutro_claro = (np.abs(imagen[..., 0] - imagen[..., 2]) < 18) & (luminancia > 150)
    texto = ndi.binary_dilation(ndi.binary_opening(neutro_claro, np.ones((9, 9))), iterations=6)
    util &= ~texto                                                     # etiqueta con el nombre

    rojo, azul = imagen[..., 0], imagen[..., 2]
    rio = util & (tophat > umbral_tophat) & (azul - rojo > -8)

    diferencia = (imagen - fondo)[rio]
    fondo_rio = fondo[rio]
    dir_pixel = diferencia / np.linalg.norm(diferencia, axis=1, keepdims=True).clip(1e-3)
    mejor = np.full(len(diferencia), -2.0)
    indice = np.zeros(len(diferencia), int)
    for i, color in enumerate(paleta):
        dir_color = color[None] - fondo_rio
        dir_color /= np.linalg.norm(dir_color, axis=1, keepdims=True).clip(1e-3)
        similitud = (dir_pixel * dir_color).sum(1)
        mejora = similitud > mejor
        mejor[mejora] = similitud[mejora]
        indice[mejora] = i

    volumen = np.full((alto, ancho), np.nan, np.float32)
    volumen[rio] = valores[indice]
    return imagen.astype(np.uint8), rio, volumen

## Estadísticas por río

In [ ]:
filas, resultados = [], {}
for ruta in capturas:
    imagen, rio, volumen = analizar(ruta)
    resultados[ruta.stem] = (imagen, rio, volumen)
    v = volumen[rio]
    ys, xs = np.nonzero(rio)
    filas.append({
        'captura': ruta.stem,
        'porcentaje_rio': rio.mean() * 100,
        'volumen_medio': v.mean(),
        'volumen_mediana': np.median(v),
        'volumen_desvio': v.std(),
        'p90': np.percentile(v, 90),
        'fraccion_bajo': (v < 1/3).mean(),
        'fraccion_normal': ((v >= 1/3) & (v < 2/3)).mean(),
        'fraccion_alto': (v >= 2/3).mean(),
        'centroide_x': xs.mean() / rio.shape[1],
        'centroide_y': ys.mean() / rio.shape[0],
    })
tabla = pd.DataFrame(filas).set_index('captura')
tabla.round(3)

## Revisión visual de la detección

Los píxeles detectados como río se pintan según su volumen (oscuro es bajo, amarillo es alto) sobre la imagen oscurecida. Es lo primero que conviene mirar para ajustar `umbral_tophat`.

In [ ]:
fig, ejes = plt.subplots(len(resultados), 1, figsize=(16, 9 * len(resultados)))
for eje, (nombre, (imagen, rio, volumen)) in zip(ejes, resultados.items()):
    vista = (imagen * 0.35).astype(np.uint8)
    vista[rio] = (plt.cm.plasma(volumen[rio])[:, :3] * 255).astype(np.uint8)
    eje.imshow(vista); eje.set_title(nombre); eje.axis('off')
plt.tight_layout()

## Perfil horizontal

Cantidad de río y volumen medio por columna, de izquierda a derecha. Puede servir para el paneo estéreo de la sonificación.

In [ ]:
fig, ejes = plt.subplots(len(resultados), 1, figsize=(14, 2.6 * len(resultados)), sharex=True)
for eje, (nombre, (imagen, rio, volumen)) in zip(ejes, resultados.items()):
    cantidad = rio.sum(0)
    con_rio = np.where(cantidad > 0, cantidad, np.nan)
    medio = np.nansum(np.where(rio, volumen, 0), 0) / con_rio
    eje.fill_between(np.arange(len(cantidad)), cantidad / cantidad.max(), alpha=.3, label='cantidad de río')
    eje.plot(pd.Series(medio).rolling(60, min_periods=10).mean(), label='volumen medio')
    eje.set_ylim(0, 1); eje.set_title(nombre); eje.legend(loc='upper right')
plt.tight_layout()

In [ ]:
tabla.to_csv('estadisticas.csv')

## Lo que falta

El top-hat todavía deja pasar algunos bordes que no son río y pierde parte del cauce principal cuando es blanco puro. Hay que ajustar los umbrales mirando las superposiciones. Como las cinco capturas son de meses distintos y casi todas de caudal bajo, la comparación entre ríos todavía no dice mucho. El siguiente paso natural es correr `analizar` sobre los 12 meses de cada río.